# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. We'll extract metadata, explore record sets, and conduct exploratory analysis and visualization, referencing all dataset structures using their `@id`s.

### Dataset Source
The dataset source is a Croissant schema hosted at:
- https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. This section loads the Croissant metadata and lets us inspect the high-level description, name, and license of the dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)
# Access the metadata object
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"License: {metadata.license}\n")

## 2. Data Overview

Explore the available record sets and their fields. All entities are referenced by their `@id`. This gives an overview of how the data is organized and stored in tables (record sets), and what fields (columns) are present.

We'll print the list of available record set `@id`s, then for each record set, print its label, description, and available field/column `@id`s.

In [ ]:
# List all available record sets and their fields, showing @id for each
print("Record Sets in dataset (with @id):\n")
record_sets = dataset.record_sets

record_set_ids = []
for rs in record_sets:
    print(f"@id: {rs['@id']}")
    print(f"  Name: {rs.get('name')}")
    print(f"  Description: {rs.get('description')}")
    fields = rs.get('field', [])
    # Normalize field format (could be list or single dict)
    if isinstance(fields, dict):
        fields = [fields]
    print("  Field @ids:")
    for field in fields:
        if isinstance(field, dict) and '@id' in field:
            print(f"    - {field['@id']}")
        elif isinstance(field, str):
            print(f"    - {field}")
    print()
    record_set_ids.append(rs['@id'])

if not record_set_ids:
    print("No record sets were detected. This schema may be describing resources outside the main record set array, or may not follow the standard Croissant 1.0 structure.")

## 3. Data Extraction

This section attempts to load data from the available record sets into pandas DataFrames for analysis. Use the `@id` for each record set to extract the data. If the dataset contains multiple record sets, each will be loaded separately.

- **Note:** If no record set is present, this section will not load any tabular data.

In [ ]:
# Attempt to extract data from each record set using its @id
dataframes = {}
if len(record_set_ids) > 0:
    for record_set_id in record_set_ids:
        print(f"Attempting to load records from record set: {record_set_id}")
        try:
            records = list(dataset.records(record_set=record_set_id))
            if records:
                df = pd.DataFrame(records)
                dataframes[record_set_id] = df
                print(f"Loaded with {len(df)} records. Columns:\n{df.columns.tolist()}\n")
            else:
                print(f"No records found for record set {record_set_id}.\n")
        except Exception as e:
            print(f"Failed to load record set {record_set_id} due to error: {e}\n")
else:
    print("No record sets detected, so no data table loaded.")

## 4. Exploratory Data Analysis (EDA)

Let's select a record set and field (referenced by `@id`) for processing. We'll filter records by a chosen numeric field, normalize its distribution, and, if categorical field(s) are available, group by them. Please update the variables below according to the printed overview.

In [ ]:
# -- USER CONFIGURABLE: Set these based on overview output --
# Example values; replace with actual '@id's and column names as detected in your context.

# Choose a record set @id to work with
if dataframes:
    record_set_id = list(dataframes.keys())[0]

    df = dataframes[record_set_id]

    # Print column names so user can choose
    print(f"Available fields (columns) for '{record_set_id}':\n{df.columns.tolist()}\n")

    # Try to automatically select numeric field and grouping field if available
    numeric_field = None
    for col in df.columns:
        # Try to pick a likely numeric column
        if df[col].dtype in [int, float] or pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if not numeric_field:
        # fallback: select the first column
        numeric_field = df.columns[0]

    group_field = None
    # Try to find a likely categorical field
    for col in df.columns:
        if df[col].dtype == object and col != numeric_field:
            group_field = col
            break
    if not group_field:
        group_field = df.columns[0] if df.columns[0] != numeric_field else (
            df.columns[1] if len(df.columns) > 1 else df.columns[0]
        )

    print(f"Selected numeric field: {numeric_field}")
    print(f"Selected group field: {group_field}\n")

    # Convert numeric field to float if possible
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    threshold = df[numeric_field].mean()  # Or set your own threshold
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
    print(filtered_df[[numeric_field]].head())

    # Normalize selected numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by group_field and calculate mean if possible
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field, f"{numeric_field}_normalized"].mean()
        print(f"\nGrouped data by {group_field} (mean values):")
        print(grouped_df.head())
else:
    print("No DataFrames loaded. Skipping EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. We'll generate a histogram and, if grouping is available, a boxplot aggregating by the group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and len(filtered_df) > 1:
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group if available
    if group_field in filtered_df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No records to visualize.")

## 6. Conclusion

In this notebook, we loaded and explored the metadata and tabular structure of the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset using `mlcroissant`. All data entities were referenced by their Croissant `@id` identifiers, providing transparency and reproducibility.

Key steps included:
- Accessing and describing record sets, fields, and columns using `@id`
- Loading available record sets into pandas DataFrames
- Applying filtering, normalization, and aggregation as typical EDA procedures
- Visualizing field distributions and grouped summaries

This approach can be easily extended to more complex multi-table datasets and custom analyses. For more details on the Croissant format and `mlcroissant`, see:
- [mlcroissant Documentation](https://mlcommons.github.io/croissant/)
- [Croissant schema specification](https://mlcommons.github.io/croissant/schema/)
